In [ ]:
import tkinter as tk
from tkinter import messagebox, ttk
import requests
import numpy as np
from sklearn.ensemble import IsolationForest
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import matplotlib.pyplot as plt
from PIL import Image, ImageTk
import io

API_KEY = 'Jf3gkg0LqSQhjJPyj4tvxy7UIuFyivVPkgxqT6fo'

# Function to generate synthetic temperature data
def generate_sensor_data(num_points=100):
    return np.random.normal(25, 5, num_points).tolist()

# Function to fetch satellite image using NASA API
def fetch_satellite_image(lat, lon):
    url = f"https://api.nasa.gov/planetary/earth/assets"
    params = {
        'lon': lon,
        'lat': lat,
        'dim': 0.1,
        'api_key': API_KEY
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        if 'url' in data:
            return data['url']
    return None



# Function to detect anomalies using Isolation Forest
def detect_anomalies(data):
    model = IsolationForest(contamination=0.1, random_state=42)
    model.fit(np.array(data).reshape(-1, 1))
    predictions = model.predict(np.array(data).reshape(-1, 1))
    anomalies = [i for i, val in enumerate(predictions) if val == -1]
    return anomalies

# Function to visualize the data
def plot_data(data, anomalies):
    fig, ax = plt.subplots()
    ax.plot(data, label='Sensor Data')
    ax.scatter(anomalies, [data[i] for i in anomalies], color='r', label='Anomalies')
    ax.set_xlabel('Time')
    ax.set_ylabel('Temperature')
    ax.legend()
    ax.grid(True)
    return fig

# GUI Application
class SensorWebApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Sensor Web Project")
        self.geometry("800x800")
        self.configure(bg='#f0f0f0')

        # Variables
        self.sensor_data = generate_sensor_data()
        self.anomalies = []

        # Create scrollable frame
        self.create_scrollable_frame()
        
        # Add widgets to the scrollable frame
        self.create_widgets()

    def create_scrollable_frame(self):
        # Create a canvas widget
        self.canvas = tk.Canvas(self, bg='#f0f0f0')
        self.scrollbar = tk.Scrollbar(self, orient="vertical", command=self.canvas.yview)
        self.scrollable_frame = tk.Frame(self.canvas, bg='#f0f0f0')

        # Configure the scrollable frame
        self.scrollable_frame.bind(
            "<Configure>",
            lambda e: self.canvas.configure(scrollregion=self.canvas.bbox("all"))
        )

        self.canvas.create_window((0, 0), window=self.scrollable_frame, anchor="nw")
        self.canvas.configure(yscrollcommand=self.scrollbar.set)

        self.canvas.pack(side="left", fill="both", expand=True)
        self.scrollbar.pack(side="right", fill="y")

    def create_widgets(self):
        # Button to generate data and detect anomalies
        self.btn_analyze = tk.Button(self.scrollable_frame, text="Generate & Analyze Data", command=self.analyze_data)
        self.btn_analyze.pack(pady=20)

        # Canvas to plot the data
        self.data_canvas = tk.Canvas(self.scrollable_frame, width=600, height=300)
        self.data_canvas.pack()

        # Button to fetch satellite image
        self.btn_fetch_image = tk.Button(self.scrollable_frame, text="Fetch Satellite Image", command=self.fetch_image)
        self.btn_fetch_image.pack(pady=20)

        # Label to display satellite image
        self.image_label = tk.Label(self.scrollable_frame)
        self.image_label.pack(pady=10)

    def analyze_data(self):
        # Detect anomalies
        self.anomalies = detect_anomalies(self.sensor_data)
        
        # Plot data with anomalies
        fig = plot_data(self.sensor_data, self.anomalies)
        canvas = FigureCanvasTkAgg(fig, master=self.data_canvas)
        canvas.draw()
        canvas.get_tk_widget().pack()

        messagebox.showinfo("Analysis Complete", f"Detected {len(self.anomalies)} anomalies.")

    def fetch_image(self):
        if not self.anomalies:
            messagebox.showwarning("No Anomalies", "No anomalies detected to fetch satellite data.")
            return

        # Fetch satellite image for a specific location
        print("Satellite image of Gwadar, Pakistan")
        lat, lon = 25.1313, 62.3250
        image_url = fetch_satellite_image(lat, lon)

        if image_url:
            response = requests.get(image_url)
            image_data = Image.open(io.BytesIO(response.content))
            image_data.thumbnail((500, 400))
            image = ImageTk.PhotoImage(image_data)
            self.image_label.configure(image=image)
            self.image_label.image = image
        else:
            messagebox.showerror("Error", "Failed to fetch satellite image.")

# Run the application
if __name__ == "__main__":
    app = SensorWebApp()
    app.mainloop()
